# S3. 이상 탐지 (온라인 보충)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/2038prob/defense-ai-textbook-code/blob/main/notebooks/online/%EC%98%A8%EB%9D%BC%EC%9D%B8%EB%B3%B4%EC%B6%A9_S3_%EC%9D%B4%EC%83%81_%ED%83%90%EC%A7%80.ipynb)

「국방 인공지능」 실습 코드 — 이 장의 파이썬 코드를 순서대로 모았습니다. **맨 위 setup 셀부터 위에서 아래로** 실행하세요(앞 셀에서 정의한 변수를 뒤 셀이 이어 씁니다). ⚠️ 표시가 붙은 셀은 개념 설명용 조각으로, 그대로는 실행되지 않을 수 있습니다.

전체 저장소: https://github.com/2038prob/defense-ai-textbook-code


In [ ]:
# === 실습 환경 준비 (Colab/로컬 공통) — 가장 먼저 실행하세요 ===
# 필요한 패키지를 설치하고(이미 있으면 건너뜀) 공통 유틸을 정의합니다.
import importlib.util, subprocess, sys
for pkg in ["torch", "torchvision", "numpy", "matplotlib", "scipy"]:
    if importlib.util.find_spec(pkg) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg])

import random, numpy as np, torch

def set_seed(seed=42):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

set_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PyTorch", torch.__version__, "| device:", device)

## AUROC와 PR-AUC: 임계값에 무관한 평가

### 🔬 실습 — AUROC vs PR-AUC: 불균형이 심해질수록 무엇이 달라지는가

① **합성 점수 만들기**: 정상 점수는 낮게, 이상 점수는 높게(겹침 있음) 생성한다.


In [ ]:
import numpy as np
from sklearn.metrics import roc_auc_score, average_precision_score

rng = np.random.default_rng(0)

def make_scores(n_normal, n_anomaly):
    s_norm = rng.normal(0.0, 1.0, n_normal)      # 정상: 평균 0
    s_anom = rng.normal(2.0, 1.0, n_anomaly)     # 이상: 평균 2 (겹침 존재)
    scores = np.concatenate([s_norm, s_anom])
    labels = np.concatenate([np.zeros(n_normal), np.ones(n_anomaly)])
    return scores, labels

# 판정기 성능(점수 분포)은 동일, '이상 비율'만 10% -> 1% -> 0.1%로
for n_anom in [1000, 100, 10]:
    scores, labels = make_scores(10000, n_anom)
    auroc = roc_auc_score(labels, scores)
    prauc = average_precision_score(labels, scores)
    print(f"이상 {n_anom/10000:>5.1%} | AUROC {auroc:.3f} | PR-AUC {prauc:.3f}")

**실행 결과 예시**

```
이상 10.0% | AUROC 0.919 | PR-AUC 0.647 \\
이상 {0}1.0% | AUROC 0.916 | PR-AUC 0.263 \\
이상 {0}0.1% | AUROC 0.897 | PR-AUC 0.216
```


## 실습: 오토인코더 이상 탐지

### 🔬 실습 — AE로 합성 데이터 이상 탐지

① **AE 모델 정의**: 인코더로 병목까지 압축, 디코더로 복원.


In [ ]:
import torch, torch.nn as nn

class AutoEncoder(nn.Module):
    def __init__(self, in_dim, z_dim=4):
        super().__init__()
        self.enc = nn.Sequential(nn.Linear(in_dim, 16), nn.ReLU(),
                                 nn.Linear(16, z_dim))          # 병목
        self.dec = nn.Sequential(nn.Linear(z_dim, 16), nn.ReLU(),
                                 nn.Linear(16, in_dim))
    def forward(self, x):
        return self.dec(self.enc(x))

② **정상 데이터로만 학습**: 재구성 손실(MSE) 최소화.


In [ ]:
import torch.optim as optim
model = AutoEncoder(in_dim=8, z_dim=4)
criterion, optimizer = nn.MSELoss(), optim.Adam(model.parameters(), lr=1e-3)
X_normal = torch.randn(2000, 8)                    # 정상 데이터(합성)
for epoch in range(50):
    optimizer.zero_grad()
    loss = criterion(model(X_normal), X_normal)    # 입력=정답 (자기 복원)
    loss.backward(); optimizer.step()

③ **임계값 설정**: 정상 데이터 재구성 오차의 99 백분위수.


In [ ]:
model.eval()
with torch.no_grad():
    err = ((model(X_normal) - X_normal) ** 2).mean(dim=1)
tau = torch.quantile(err, 0.99)                    # 정상 99% 포함
print(f"임계값 tau = {tau:.4f}")

④ **이상 판정**: 새 데이터의 재구성 오차가 임계값 τ를 넘으면 이상.


In [ ]:
X_test = torch.randn(100, 8)
X_test[:5] += 5.0                                  # 앞 5개를 인위적 이상으로
with torch.no_grad():
    test_err = ((model(X_test) - X_test) ** 2).mean(dim=1)
is_anomaly = test_err > tau
print("이상 탐지 수:", is_anomaly.sum().item())

## 국방 적용과 개념 드리프트

### 🔬 실습 — 개념 드리프트를 겪어 보기: 고정 임계값 vs 적응형 임계값

① **드리프트가 있는 "전부 정상" 시계열 만들기.** 전반부는 평균 0.5, 후반부는 계절 변화로 평균이 0.7로 *서서히* 이동한다(예: 하절기 장비 온도 기준선 상승).


In [ ]:
import numpy as np
rng = np.random.default_rng(7)

T = 1000
base = 0.5 + 0.1 * rng.standard_normal(500)          # t=0~499: 기존 정상
alpha = np.linspace(0, 1, 500)                        # 0 -> 1 서서히
drift = 0.5 + 0.2 * alpha + 0.1 * rng.standard_normal(500)  # 평균 0.5->0.7
y = np.concatenate([base, drift])                     # 전 구간 '정상'

② **고정 임계값.** 전반부(학습 구간) 통계로 τ = μ + 3σ를 정하고 후반부에 그대로 적용한다.


In [ ]:
mu, sigma = y[:500].mean(), y[:500].std()
tau_fixed = mu + 3 * sigma
fa_fixed = (y[500:] > tau_fixed).mean()
print(f"고정 임계값 {tau_fixed:.3f} | 후반부 오경보율 {fa_fixed:.1%}")

**실행 결과 예시**

```
고정 임계값 0.768 | 후반부 오경보율 8.0%
```


③ **적응형 임계값.** 운용자가 정상으로 확인한 *최근* 관측 200개로 통계를 계속 갱신한다(슬라이딩 윈도우 재적합).


In [ ]:
W = 200                                 # 최근 정상 윈도우 크기
fa_count = 0
for t in range(500, T):
    recent = y[t-W:t]                   # 최근 W개 (운용상 '확인된 정상' 가정)
    tau_t = recent.mean() + 3 * recent.std()
    fa_count += (y[t] > tau_t)
print(f"적응형 | 후반부 오경보율 {fa_count/500:.1%}")

**실행 결과 예시**

```
적응형 | 후반부 오경보율 0.4%
```


---

실습 코드 저장소: https://github.com/2038prob/defense-ai-textbook-code  ·  교재 「국방 인공지능」
